# Notebook 1: How a language model reads biological sequences

**LLMs in Bioinformatics workshop** | ~90 minutes | no GPU needed | no maths needed

By the end of this notebook you will be able to:

1. explain what a **token** is, and compare how text, protein and DNA (k-mers and BPE) are cut into tokens, and see how a text LLM cuts up gene names, rsIDs and variant names;
2. use a protein language model to **fill in a hidden amino acid** and read its confidence;
3. look at **attention** (which residues the model "looks at") and say what it can and cannot tell you;
4. test a model with a simple **control experiment** (scrambled sequence).

### Time plan (90 min)

| Time | Part | What happens |
| --- | --- | --- |
| 0:00-0:10 | **0. Setup** | run the setup cells, load the model |
| 0:10-0:30 | **1. Tokens** | how text, protein, DNA (k-mers and BPE) and gene names become tokens |
| 0:30-0:50 | **2. Masked prediction** | the "fill in the blank" game with a protein model |
| 0:50-1:10 | **3. Attention** | what the model looks at |
| 1:10-1:20 | **4. Scramble test** | does the model know real proteins from noise? |
| 1:20-1:30 | **5. Open time** | challenges, questions, troubleshooting |

Each part follows the same rhythm: **short demo (about 5 min) -> you try it yourself (8-10 min) -> debrief together (3-5 min)**.

### How to use this notebook

- Run cells top to bottom with **Shift+Enter**. Cells marked **TODO** need you to replace `...` with your own code.
- After each exercise a **check cell** tells you whether your answer is right. It is fine if it says no.
- **Try for 3 minutes before opening a hint** (click the triangle). Hints are written to give you a nudge, not the answer.
- Stuck, or something broke? Check the **Troubleshooting** table at the bottom, then ask.
- Write your guesses in the text cells *before* you run the code. Predicting first is how you learn the most.

## Part 0: Setup (about 10 min)

We will use **ESM-2 (8M parameters)**, a small protein language model from Meta's Evolutionary Scale Modeling project. It is public on Hugging Face and runs on a plain CPU, so you do not need a GPU.

In [ ]:
# Install and import. (Colab already has most of these; this is quick.)
!pip install -q "transformers>=4.40" matplotlib

import numpy as np
import torch
import matplotlib.pyplot as plt
import transformers
from transformers import AutoTokenizer, EsmForMaskedLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| torch", torch.__version__, "| device:", device)

In [ ]:
# Load the protein model and its tokenizer from Hugging Face (first run downloads ~35 MB).
MODEL = "facebook/esm2_t6_8M_UR50D"

tok = AutoTokenizer.from_pretrained(MODEL)
try:
    # "eager" attention lets us read the attention weights later (Part 3)
    model = EsmForMaskedLM.from_pretrained(MODEL, attn_implementation="eager")
except Exception:
    model = EsmForMaskedLM.from_pretrained(MODEL)
model = model.eval().to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Loaded {MODEL} with {n_params/1e6:.1f} million parameters")

In [ ]:
# Our test protein for the whole notebook: human ubiquitin (76 amino acids),
# a small, very well studied protein.
ubq = "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
assert len(ubq) == 76
print(ubq)

**Your guess (double-click to edit):** In one sentence, what do you think a language model trained only on millions of protein sequences could learn?

> _write your guess here_

## Part 1: Tokens (about 20 min, including Part 1b)

A language model cannot read letters or words directly. Text is first cut into pieces called **tokens**, each token gets a number (an ID), and the model works on those numbers.

Different kinds of sequences are cut up in different ways. Let's compare three.

In [ ]:
# English text: tokens are often whole words, but rare words are split into pieces.
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
sentence = "Bioinformatics uses transformers."
print(bert_tok.tokenize(sentence))

In [ ]:
# Protein: one token per amino acid, plus special start/end tokens.
enc = tok(ubq)
tokens = tok.convert_ids_to_tokens(enc["input_ids"])
print("first tokens:", tokens[:8])
print("last tokens: ", tokens[-3:])
print("their IDs:   ", enc["input_ids"][:8])

In [ ]:
# DNA: there is no single standard. Some models use k-mers (chunks of k bases).
def kmers(seq, k=3, step=1):
    # step=1: overlapping chunks, step=k: non-overlapping chunks
    return [seq[i:i + k] for i in range(0, len(seq) - k + 1, step)]

dna = "ATGGCGTACGTT"
print("overlapping 3-mers:    ", kmers(dna, k=3, step=1))
print("non-overlapping 3-mers:", kmers(dna, k=3, step=3))

### Exercise 1: count the tokens (TODO)

**1a.** How many tokens does the ESM tokenizer give for ubiquitin (76 residues)? Use `enc` from above. Before running, write your guess: ______

**1b.** A 30-base DNA sequence is cut into **6-mers**. How many tokens do you get with overlapping chunks (`step=1`) and with non-overlapping chunks (`step=6`)? Use `kmers()`.

**1c.** Every tokenizer has a special **mask token** (the "blank" in fill-in-the-blank). Find its ID. We need it in Part 2.

In [ ]:
# 1a
n_tokens = ...        # TODO

# 1b
dna30 = "ATGGCGTACGTTAGCCTAGGATCCAATGCA"
assert len(dna30) == 30
n_overlap = ...       # TODO
n_nonoverlap = ...    # TODO

# 1c
mask_id = ...         # TODO

<details><summary>Hint 1a/1b</summary>

`enc["input_ids"]` is a list. `len(...)` counts its items. For 1b, `kmers(dna30, k=6, step=1)` returns a list of chunks.
</details>

<details><summary>Hint 1c</summary>

The tokenizer object `tok` has an attribute called `mask_token_id`. You can also print `tok.all_special_tokens` to see the special tokens.
</details>

In [ ]:
# Check your answers
assert n_tokens is not ..., "1a: replace the ... with your code"
assert n_tokens == 78, f"1a: expected 78 (76 residues + 2 special tokens), got {n_tokens}"
assert n_overlap == 25 and n_nonoverlap == 5, f"1b: expected 25 and 5, got {n_overlap} and {n_nonoverlap}"
assert mask_id is not ... and tok.convert_ids_to_tokens(mask_id) == "<mask>", "1c: that ID is not the <mask> token"
print("Correct! 1a, 1b, 1c all pass.")

**Debrief questions (discuss with your neighbour):**

- Where do the 2 extra protein tokens come from?
- The same 30 bases gave 25 tokens or 5 tokens depending on how we chunked them. A model can only read a limited number of tokens at once (its *context length*). Which chunking lets it read a longer stretch of genome? What might you lose?
- Some DNA models use **one token per base** and still manage very long contexts. What kind of change would that need?

## Part 1b: tokenizers you will meet in practice (about 10 min)

Two more tokenizers matter for genomics work.

**1. DNA with byte-pair encoding (BPE).** *k-mers* cut DNA into chunks of one fixed size. **DNABERT-2** instead learns a vocabulary of frequent DNA chunks from the genome itself (BPE, the same idea used by text LLMs). A token can be 2, 4 or 7 bases long, so repeated motifs become single tokens and a fixed-length context covers more DNA.

**2. A general text LLM (the kind behind chat assistants).** Its tokenizer was built from web text, not biology. Gene names, rsIDs and variant names (HGVS) are cut into arbitrary pieces. The model never sees "this is a gene" or "this is a variant"; it sees fragments. Run the cells and look.

In [ ]:
# DNABERT-2: learned DNA "words" (BPE). Only the small tokenizer files are downloaded.
try:
    dna_bpe = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-2-117M")
except Exception as e:
    dna_bpe = None
    print("Could not load the DNABERT-2 tokenizer:", type(e).__name__, "- the BPE lines are skipped, the notebook continues.")

dna_demo = "ATGGCGTACGTTAGCCTAGGATCCAATGCAGGCTTAAGGCCTTAACGGATCCGATCGGATTACA"
if dna_bpe is not None:
    pieces = dna_bpe.tokenize(dna_demo)
    print(f"{len(dna_demo)} bases -> {len(pieces)} BPE tokens")
    print(pieces)
    print("bases per token, BPE:                     ", round(len(dna_demo) / len(pieces), 2))
print("bases per token, overlapping 6-mers:      ", round(len(dna_demo) / len(kmers(dna_demo, 6, 1)), 2))
print("bases per token, non-overlapping 6-mers:  ", round(len(dna_demo) / len(kmers(dna_demo, 6, 6)), 2))

In [ ]:
# A general text tokenizer (GPT-2) on biological names. "Ġ" marks the start of a word after a space.
gpt_tok = AutoTokenizer.from_pretrained("gpt2")
bio_strings = ["BRCA1", "BRCA2", "brca1", "rs80357906", "NM_007294.4", "p.Cys61Gly", "c.5266dupC",
               "NM_007294.4(BRCA1):c.5266dupC", "C61G"]
for s in bio_strings:
    pieces = gpt_tok.tokenize(s)
    print(f"{s:32s} {len(pieces):2d} tokens  {pieces}")

### Exercise 1b: how badly is each name cut up? (TODO)

**1d.** For every string in `bio_strings`, compute **tokens per character** (number of GPT-2 tokens divided by the length of the string). Store it in a dictionary `{string: value}` called `tokens_per_char`. A value near 1 means "almost one token per character", a small value means "long, whole pieces".

**1e.** Which string has the **highest** value? Store it in `most_fragmented`. Before you run it, write your guess: ______

Then look at the output of the previous cell again: do `BRCA1` and `BRCA2` get the same number of tokens? Does `brca1` (lowercase) look like `BRCA1` to the model?

In [ ]:
# 1d
tokens_per_char = ...      # TODO: a dict {string: tokens / characters}

# 1e
most_fragmented = ...      # TODO: the string with the largest value

<details><summary>Hint 1d/1e</summary>

A dictionary comprehension looks like `{s: f(s) for s in my_list}`. `gpt_tok.tokenize(s)` gives the list of pieces, so `len(...)` of it is the token count. For 1e, `max(d, key=d.get)` returns the key with the largest value.
</details>

In [ ]:
# Check
assert tokens_per_char is not ... and isinstance(tokens_per_char, dict), "1d: build a dictionary"
assert set(tokens_per_char) == set(bio_strings), "1d: one entry per string in bio_strings"
assert all(0 < v <= 1 for v in tokens_per_char.values()), "1d: values should be tokens divided by characters (between 0 and 1)"
assert most_fragmented is not ... and most_fragmented == max(tokens_per_char, key=tokens_per_char.get), "1e: pick the string with the largest value"
print("Correct! The most fragmented string was:", most_fragmented, f"({tokens_per_char[most_fragmented]:.2f} tokens per character)")

**Debrief questions:**

- Compare bases per token for BPE, overlapping 6-mers and non-overlapping 6-mers. Which lets a fixed context window cover the most DNA, and which keeps the most fine detail (single-base changes)?
- To a text LLM, `rs80357906` is a handful of arbitrary pieces. It cannot *look the variant up*; it can only recall whatever association it formed between those pieces during training. What kind of error does that invite when you ask it for a gene, rsID or ClinVar classification? (This comes back in **Notebook 5**.)
- The same protein variant can be written `C61G`, `p.Cys61Gly` or `p.(Cys61Gly)`. A model sees each as a different token sequence. Why is it safer to **normalise names in code** (as in Notebook 2) before sending anything to a model?

## Part 2: Masked prediction, "fill in the blank" (about 20 min)

ESM-2 was trained with one simple game: hide some amino acids in a real protein and ask the model to guess them from the rest. To win, it has to pick up what real proteins look like (which residues tend to go together, which positions are conserved).

The model does not answer with one letter. It gives a **probability for each of the 20 amino acids** (plus a few special tokens). Let's look.

In [ ]:
def masked_predictions(seq, pos, top=5):
    """Hide the residue at 0-based position `pos`, return the model's top guesses and all probabilities."""
    enc = tok(seq, return_tensors="pt").to(device)
    ids = enc["input_ids"].clone()
    ids[0, pos + 1] = tok.mask_token_id          # +1 because position 0 is the <cls> token
    with torch.no_grad():
        logits = model(input_ids=ids, attention_mask=enc["attention_mask"]).logits
    probs = logits[0, pos + 1].softmax(-1)        # turn scores into probabilities that sum to 1
    top_p, top_i = probs.topk(top)
    guesses = [(tok.convert_ids_to_tokens(int(i)), round(float(p), 3)) for p, i in zip(top_p, top_i)]
    return guesses, probs

for pos in [3, 45, 70]:
    guesses, _ = masked_predictions(ubq, pos)
    print(f"position {pos} (true residue {ubq[pos]}):", guesses)

### Exercise 2a: pick a position and predict first (TODO)

Choose any position `my_pos` between 0 and 75. **Before running**, write your guess: will the model put the true residue first? How confident will it be?

> _your guess_

Then get the probability the model gave to the **true** residue.

In [ ]:
my_pos = ...           # TODO: choose a number from 0 to 75
guesses, probs = masked_predictions(ubq, my_pos)
print("true residue:", ubq[my_pos], "| model's top guesses:", guesses)

true_id = tok.convert_tokens_to_ids(ubq[my_pos])   # the token ID of the true residue
p_true_here = ...      # TODO: the probability the model gave to the true residue
print("probability of the true residue:", p_true_here)

<details><summary>Hint</summary>

`probs` is a list of probabilities, one per token ID. Look up the entry that belongs to `true_id`, like `probs[...]`. Wrap it in `float(...)` to get a plain number.
</details>

In [ ]:
# Check
assert my_pos is not ... and 0 <= my_pos <= 75, "choose my_pos between 0 and 75"
assert p_true_here is not ... and 0.0 <= float(p_true_here) <= 1.0, "p_true_here should be a probability between 0 and 1"
print("OK, probability =", round(float(p_true_here), 3))

### Exercise 2b: scan the whole protein (TODO)

Now repeat this for **every** position: hide one residue at a time and record the probability of the true residue. Fill in the one TODO line.

In [ ]:
p_true = []
for i in range(len(ubq)):
    _, probs = masked_predictions(ubq, i)
    p_true.append(...)       # TODO: probability of the true residue ubq[i]

<details><summary>Hint</summary>

Same idea as Exercise 2a: `tok.convert_tokens_to_ids(ubq[i])` gives the ID, and `probs[...]` looks up the probability.
</details>

In [ ]:
# Check, then plot (the plot is provided)
assert len(p_true) == 76 and all(isinstance(p, float) for p in p_true), "p_true should be a list of 76 numbers (use float(...))"

hydrophobic = set("AVILMFWC")
colors = ["tab:orange" if aa in hydrophobic else "tab:blue" for aa in ubq]
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.bar(range(len(ubq)), p_true, color=colors)
ax.set_xticks(range(len(ubq)))
ax.set_xticklabels(list(ubq), fontsize=7)
ax.axhline(np.mean(p_true), color="black", ls="--", lw=1, label=f"mean = {np.mean(p_true):.2f}")
ax.set_ylabel("P(true residue)")
ax.set_xlabel("position (orange = hydrophobic residue, blue = other)")
ax.legend()
plt.tight_layout()
plt.show()

**Debrief questions:**

- Which positions does the model find easy, which hard? Is there a pattern in the residue types (orange vs blue)?
- The model was never shown a 3D structure and never told what ubiquitin is. What do you think it has learned that lets it guess some positions?
- A low probability does not mean the model is wrong about biology. Can you think of a reason a residue might legitimately be hard to predict? (Hint: think about what varies between species.)

## Part 3: Attention, what does the model look at? (about 20 min)

When the model processes a sequence, every residue "looks at" every other residue and decides how much each one matters for understanding it. The strength of that looking is the **attention weight** (between 0 and 1).

- One **head** is one learned way of looking (think of one highlighter colour).
- A **layer** has many heads working side by side; the model stacks several layers.
- ESM-2 8M has **6 layers x 20 heads = 120 heads**.

Let's pull the attention weights for ubiquitin.

In [ ]:
enc = tok(ubq, return_tensors="pt").to(device)
with torch.no_grad():
    out = model(**enc, output_attentions=True)

assert out.attentions is not None and out.attentions[0] is not None, \
    "No attention returned. See Troubleshooting: 'attentions are None'."

attn = torch.stack(out.attentions).squeeze(1)       # (layers, heads, L, L)
A = attn[:, :, 1:-1, 1:-1]                          # drop <cls> and <eos>: only real residues
A = (A / A.sum(-1, keepdim=True)).cpu().numpy()     # make each row sum to 1 again
print("attention array shape (layers, heads, residue, residue):", A.shape)

In [ ]:
def plot_head(layer, head):
    # Rows = the residue that is "looking"; columns = the residue it looks at.
    fig, ax = plt.subplots(figsize=(5.5, 4.8))
    im = ax.imshow(A[layer, head], cmap="viridis")
    ax.set_xlabel("residue being looked at")
    ax.set_ylabel("residue that is looking")
    ax.set_title(f"layer {layer + 1}, head {head + 1}")
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.show()

def plot_layer(layer):
    # All 20 heads of one layer
    fig, axes = plt.subplots(4, 5, figsize=(12, 9))
    for h, ax in enumerate(axes.ravel()):
        ax.imshow(A[layer, h], cmap="viridis")
        ax.set_title(f"head {h + 1}", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"Layer {layer + 1}: all 20 heads")
    plt.tight_layout()
    plt.show()

plot_head(layer=5, head=0)     # one head from the last layer
plot_layer(layer=5)

**Look at the pictures before reading on.** Some heads show a bright diagonal (each residue looks at its neighbours). Others look blotchy or striped. Different heads do different things.

### Exercise 3: which heads look nearby and which look far? (TODO)

For each head, compute the **average attention distance**: for every residue `i`, take the attention it gives to each residue `j`, multiply by the sequence distance `|i - j|`, add up over `j`, then average over `i`.

- small number = the head mostly looks at **neighbours**
- large number = the head looks at **distant** residues

In [ ]:
idx = np.arange(A.shape[-1])
dist = np.abs(idx[:, None] - idx[None, :])     # |i - j| for every pair of residues, shape (76, 76)

mean_dist = ...        # TODO: array of shape (6, 20), one number per head

<details><summary>Hint</summary>

`A` has shape (layers, heads, i, j). `A * dist` multiplies every head's matrix by the distances. Then add up over the last axis (`.sum(-1)`, that is the sum over j) and average over the next one (`.mean(-1)`, the average over i).
</details>

In [ ]:
# Check, then plot the result (provided)
assert mean_dist is not ... and mean_dist.shape == (6, 20), "mean_dist should have shape (6, 20)"
fig, ax = plt.subplots(figsize=(8, 3.5))
im = ax.imshow(mean_dist, cmap="magma", aspect="auto")
ax.set_xlabel("head"); ax.set_ylabel("layer")
ax.set_xticks(range(20)); ax.set_xticklabels(range(1, 21))
ax.set_yticks(range(6)); ax.set_yticklabels(range(1, 7))
plt.colorbar(im, label="mean attention distance (residues)")
plt.show()

Now find the **most local** head and the **most long-range** head, and look at them.

In [ ]:
local_layer, local_head = ...   # TODO: position of the smallest value in mean_dist
far_layer, far_head = ...       # TODO: position of the largest value in mean_dist

<details><summary>Hint</summary>

`mean_dist.argmin()` gives the position as a single number counted along the whole flattened array. `np.unravel_index(that_number, mean_dist.shape)` turns it back into (layer, head).
</details>

In [ ]:
# Check, then plot both heads
assert local_layer is not ... and far_layer is not ..., "fill in both TODO lines"
local_layer, local_head, far_layer, far_head = int(local_layer), int(local_head), int(far_layer), int(far_head)
print(f"most local head:      layer {local_layer + 1}, head {local_head + 1}  (mean distance {mean_dist[local_layer, local_head]:.1f})")
print(f"most long-range head: layer {far_layer + 1}, head {far_head + 1}  (mean distance {mean_dist[far_layer, far_head]:.1f})")
plot_head(local_layer, local_head)
plot_head(far_layer, far_head)

**Debrief questions:**

- What does the local head's picture look like, compared with the long-range head's?
- In a folded protein, residues far apart in the sequence can touch in 3D. Why might a long-range head be interesting? (Think about it, but do not conclude that it *found* the contacts. We did not test that.)
- **Important caution:** attention shows *where the model looks*, not *why it gave its answer*. Heads are not guaranteed to correspond to anything a biologist would recognize. Treat attention pictures as a hint for exploring, not as evidence.

### Optional: interactive attention viewer (bertviz)

Skip this if you are short on time or the install fails; nothing later depends on it.

In [ ]:
# OPTIONAL: interactive view of attention (hover and click on heads)
try:
    !pip install -q bertviz
    from bertviz import head_view
    head_view(out.attentions, tok.convert_ids_to_tokens(enc["input_ids"][0]))
except Exception as e:
    print("bertviz not available, skipping:", e)

## Part 4: A control experiment, the scramble test (about 10 min)

Good science needs controls. Does the model really know something about real proteins, or would it do equally well on nonsense?

**Scrambled ubiquitin** has exactly the same 76 letters (same amino acid composition), just in random order.

**Your prediction (write it before running):** will the model's average probability of the true residue go up, stay the same, or go down for the scrambled protein? Why?

> _your prediction_

In [ ]:
def p_true_all(seq):
    """Same as the loop in Exercise 2b but faster: hide each position in turn, all at once in one batch."""
    enc = tok(seq, return_tensors="pt")
    L = len(seq)
    ids = enc["input_ids"].repeat(L, 1)
    for i in range(L):
        ids[i, i + 1] = tok.mask_token_id
    mask = enc["attention_mask"].repeat(L, 1)
    with torch.no_grad():
        logits = model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
    rows = torch.arange(L)
    probs = logits[rows, rows + 1].softmax(-1)
    true_ids = enc["input_ids"][0, 1:-1].to(device)
    return probs[rows, true_ids].cpu().numpy()

# sanity check: matches the slow loop from Exercise 2b
assert np.allclose(p_true_all(ubq), p_true, atol=1e-3), "batch version disagrees with your loop from 2b"
print("batch version matches your loop from 2b")

### Exercise 4: scramble and score (TODO)

In [ ]:
rng = np.random.default_rng(0)                      # fixed seed so everyone gets the same scramble
scrambled = ...                                      # TODO: shuffle the letters of ubq and join them back into a string

p_native = ...                                       # TODO: p_true_all of the real sequence
p_scrambled = ...                                    # TODO: p_true_all of the scrambled sequence

<details><summary>Hint</summary>

`rng.permutation(list(ubq))` returns the letters in random order. `"".join(...)` glues a list of letters back into one string. Check that `sorted(scrambled) == sorted(ubq)`: same letters, new order.
</details>

In [ ]:
# Check, then compare
assert scrambled is not ... and sorted(scrambled) == sorted(ubq) and scrambled != ubq, "scrambled should use the same letters in a different order"
assert p_native is not ... and p_scrambled is not ..., "fill in both p_ lines"
print("scrambled sequence:", scrambled)
print(f"mean P(true residue)  native: {p_native.mean():.3f}   scrambled: {p_scrambled.mean():.3f}")

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(p_native, label="native ubiquitin")
ax.plot(p_scrambled, label="scrambled")
ax.set_xlabel("position"); ax.set_ylabel("P(true residue)"); ax.legend()
plt.show()

**Debrief questions:**

- Was your prediction right? What does the result tell you about what the model has learned?
- This was **one** scramble of **one** protein. How would you make the conclusion more convincing? (More scrambles? More proteins? A different kind of control?)
- The next notebooks use exactly this kind of score to judge **mutations**: if a mutation makes the model less happy with the sequence, it may be damaging.

## Part 5: Open time (about 10 min)

Use this time for whichever you need: finish earlier exercises, ask questions, troubleshoot, or try a challenge.

### Key points to remember

- **Tokens** are the pieces a model reads; the way you cut a sequence into tokens is a design choice.
- A **masked language model** learns by guessing hidden residues, and gives a *probability* for each possible answer.
- **Attention** shows where each residue looks, in many heads and layers. It is a window into the model, not an explanation.
- **Controls** (such as scrambled sequences) tell you whether a result means something.

### Challenge A: your own protein

Paste a protein sequence (from UniProt) and see where the model is confident. Up to 300 residues; a short default is provided.

In [ ]:
my_seq = "FVNQHLCGSHLVEALYLVCGERGFFYTPKT"     # default: human insulin B chain. Replace with your own!

my_seq = my_seq.strip().upper()
assert set(my_seq) <= set("ACDEFGHIKLMNPQRSTVWY"), "use only the 20 standard amino-acid letters"
assert len(my_seq) <= 300, "keep it under 300 residues to avoid running out of memory"

p = p_true_all(my_seq)
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(len(my_seq)), p)
ax.set_xticks(range(len(my_seq))); ax.set_xticklabels(list(my_seq), fontsize=7 if len(my_seq) < 100 else 4)
ax.set_ylabel("P(true residue)")
plt.show()
print(f"mean P(true residue): {p.mean():.3f}")

### More challenges

- **B. Bigger model:** change `MODEL` in Part 0 to `facebook/esm2_t12_35M_UR50D` (35M parameters) and re-run. Do the probabilities and the scramble gap change?
- **C. Hide several residues at once:** mask 5, 10, 20 positions together. How fast does the accuracy fall?
- **D. A peek at mutations (preview of the variant-effect notebooks):** hide position 45, then compare the probability the model gives to the true residue with the probability it gives to a different residue. Which is bigger?
- **E. Tokens:** write a function that turns a DNA sequence into overlapping 4-mers and counts how many distinct 4-mers can exist. (Hint: 4 letters, 4 positions.)

---

## Troubleshooting

| Problem | What to try |
| --- | --- |
| `ModuleNotFoundError` after the install cell | Run **Runtime > Restart session**, then run the cells again from the top |
| Download of the model is very slow or fails (`429`, `timeout`) | Wait one minute and re-run. If it keeps failing, add an `HF_TOKEN` in Colab Secrets (key icon on the left) and re-run |
| `CUDA out of memory` | Not expected in this notebook. **Runtime > Restart session**, or switch to CPU: Runtime > Change runtime type > None |
| "No attention returned" or `attentions` are `None` | Re-run the model-loading cell in Part 0 (it asks for "eager" attention). If it still fails, restart the session and run from the top |
| A check cell fails with a `...` message | You have not replaced the `...` placeholder yet. Replace it with your code and re-run both the TODO cell and the check cell |
| `NameError: name '...' is not defined` | A cell above was not run. Use **Runtime > Run before** (Ctrl+F8) to run everything above the current cell |
| Numbers differ slightly from your neighbour's | Small differences between CPU and GPU runs are normal. Large differences usually mean a different position or model |
| Colab disconnected | Re-run from the top. The model files are cached, so the second time is faster |

**Still stuck?** Copy the full red error message, put your hand up, and show it on your screen.